# S3.1 — SparkSession
**Date completed:** September 2026  
**Status:** In Progress  
**Interview covered:** Q3 — SparkSession, Q4 — SparkContext vs SparkSession

# SparkSession exploration:

In [0]:
# ============================================================
# Cell 2 — SparkSession: Exploring the entry point
#
# Goal:
# 1. Understand what SparkSession provides
# 2. Check version and configuration
# 3. Prove SparkSession wraps SparkContext
# 4. Use all major SparkSession methods
# ============================================================

import pyspark.sql.functions as F
# ------------------------------------------------------------
# Step 1 — Verify SparkSession exists automatically
# ------------------------------------------------------------
print("=== SPARKSESSION EXPLORATION ===")
print()

# ------------------------------------------------------------
# Step 1 — Inspect the existing SparkSession
# Databricks provides the 'spark' session automatically.
# ------------------------------------------------------------

print(f"SparkSession type : {type(spark)}")
print(f"Spark version : {spark.version}")

# ------------------------------------------------------------
# Step 2 — Inspect supported session configurations
# ------------------------------------------------------------

print("=== KEY CONFIGURATIONS ===")

print("Shuffle partitions:",spark.conf.get("spark.sql.shuffle.partitions"))

print("ANSI mode:",spark.conf.get("spark.sql.ansi.enabled"))

print("Session time zone:",spark.conf.get("spark.sql.session.timeZone"))
print()

# ------------------------------------------------------------
# Step 3 — spark.range()
# Generate a DataFrame containing sequential numbers.
# ------------------------------------------------------------
print("=== spark.range() ===")

spark.range(1, 6).show()

# ------------------------------------------------------------
# Step 4 — spark.createDataFrame()
# Create a DataFrame from local Python data.
# ------------------------------------------------------------

print("=== spark.createDataFrame() ===")
sample_data = [
    (1, "Delhi", 2500.0),
    (2, "Mumbai", 1800.0),
    (3, "Pune", 3200.0)
]

df = spark.createDataFrame(
    sample_data, ["order_id", "city", "order_value"])
df.show()

# ------------------------------------------------------------
# Step 5 — spark.sql()
# Execute SQL using the SparkSession.
# ------------------------------------------------------------
print("=== spark.sql() ===")
spark.sql("""SELECT current_timestamp() AS current_time""").show()

# ------------------------------------------------------------
# Step 6 — Create a session temporary view
# Demonstrates using the same data through SQL.
# ------------------------------------------------------------

print("=== TEMP VIEW EXPLORATION ===")

df.createOrReplaceTempView("orders_demo")

spark.sql("""
    SELECT 
        city,
        sum(order_value) AS total_revenue
    FROM orders_demo
    GROUP BY CITY
""")

# ------------------------------------------------------------
# Step 7 — spark.table()
# Read the registered temporary view as a DataFrame.
# ------------------------------------------------------------

print("=== spark.table() ===")
df_from_view = spark.table("orders_demo")
df_from_view.show()


# ------------------------------------------------------------
# Step 8 — SparkSession vs SparkContext
#
# SparkContext is not directly available on Serverless.
# We demonstrate the conceptual distinction instead.
# ------------------------------------------------------------

print("=== SPARKSESSION vs SPARKCONTEXT ===")

print("SparkSession: Unified entry point for DataFrames and SQL.")
print("SparkContext: Lower-level entry point for Spark Core/RDD APIs.")
print("Databricks Serverless: Use supported SparkSession APIs.")
print()

print("=== SPARKSESSION EXPLORATION COMPLETED ===")

# S3.1 — SparkSession: Key Takeaways

## 1. What Is SparkSession?

SparkSession is the unified entry point for working with structured data in Spark, especially DataFrames and SQL.

Databricks automatically provides an existing SparkSession named `spark` in Python notebooks.

We do not need to manually create another session for our current practical.

---

## 2. Key SparkSession Properties and Methods

| Method / Property | Purpose | Example |
|---|---|---|
| spark.version | Check Spark version | spark.version |
| spark.conf.get() | Read session configuration | Check shuffle partitions |
| spark.conf.set() | Modify supported session configuration | Set shuffle partitions |
| spark.range() | Generate sequential data | spark.range(1, 100) |
| spark.createDataFrame() | Create a DataFrame | spark.createDataFrame(data, schema) |
| spark.sql() | Execute SQL queries | spark.sql("SELECT ...") |
| spark.table() | Access a table or registered view | spark.table("orders_demo") |

Note: `spark.version` is a property. Its actual value depends on the runtime.

---

## 3. Temporary View Pattern

```python
# Register an existing DataFrame as a temporary view
df.createOrReplaceTempView("orders_temp")

# Query the temporary view using SQL
spark.sql("SELECT * FROM orders_temp").show()

# Access the view through the DataFrame API
spark.table("orders_temp").show()
```

Important:
- createOrReplaceTempView() is a DataFrame method.
- Creates or replaces a session-scoped temporary view.
- Does not independently persist the underlying data.
- Can be queried using SQL.
- The view is removed when the associated session ends.

Detailed table and view types will be covered separately.

---

## 4. SparkSession vs SparkContext

| Feature | SparkSession | SparkContext |
|---|---|---|
| API level | High-level structured APIs | Lower-level Spark Core APIs |
| DataFrames | Yes | Not its primary interface |
| SQL | Yes | Not its primary interface |
| RDD operations | Not directly | Yes |
| Available in Databricks Serverless | Yes | Direct access unsupported |
| Typical usage | Data Engineering, SQL, DataFrames | Low-level Spark operations |

### Important Understanding

SparkSession provides the unified interface for modern structured Spark workloads.

In classic Spark, SparkSession provides access to an associated SparkContext.

SparkContext supports lower-level Spark functionality, including RDD operations.

Databricks Serverless uses Spark Connect, where direct SparkContext and RDD APIs are unsupported.

For our current Serverless learning, use SparkSession, DataFrames and SQL.

---

## 5. Our Configuration Findings

| Configuration | Observed / Example Value | Meaning |
|---|---|---|
| spark.sql.shuffle.partitions | auto | Databricks automatically manages shuffle partition selection |
| spark.sql.ansi.enabled | true | Enables ANSI SQL behaviour |
| spark.sql.session.timeZone | UTC | Session timezone for relevant timestamp operations |
| spark.version | Runtime-dependent | Installed Spark version |

### ANSI Mode

When ANSI mode is enabled, Spark applies stricter SQL behaviour.

For example, certain invalid type conversions or arithmetic errors can raise exceptions rather than silently returning null or unexpected results.

### Session Timezone

UTC is commonly used for consistent timestamp processing across systems.

However, a DATE column contains a calendar date without a timezone.

Timestamp and timezone handling should follow the business requirement and data type.

### Shuffle Partitions

Our Databricks Serverless environment uses:

```python
spark.conf.get("spark.sql.shuffle.partitions")
```

Observed setting: auto

Databricks can automatically optimise shuffle partitioning.

---

## 6. Interview Questions

**Q1. What is SparkSession?**

SparkSession is the unified entry point for working with structured Spark data through DataFrame and SQL APIs.

**Q2. Is SparkSession automatically available in Databricks?**

Yes. Databricks Python notebooks provide an existing SparkSession named `spark`.

**Q3. What is the difference between SparkSession and SparkContext?**

SparkSession provides high-level DataFrame and SQL functionality, while SparkContext provides access to lower-level Spark Core functionality.

**Q4. Why did spark.sparkContext fail in our practical?**

Our notebook runs on Databricks Serverless, which uses Spark Connect and does not support direct SparkContext access.

**Q5. Why do we create temporary views?**

To make an existing DataFrame accessible through SQL without creating a permanent table.

---

## Final Learning Conclusion

SparkSession is our primary entry point for PySpark development.

Through SparkSession, we can create DataFrames, execute SQL, access tables and views, and manage supported session configurations.

SparkContext is important for understanding Spark's underlying architecture, but direct SparkContext APIs are not available in our current Serverless environment.